## SHNITSEL dataset to extended xyz files
Notebook to convert the nc files from the shnitsel dataset into xyz files. 

Dependencies:
- Run pip install xarray netCDF4
- Used to read the nc file type

Conversion details: 
- nc default is written in BOHR (distance) and HARTREE (energy)
- for xyz files convention is default Angstrom 
- X-MACE dataset used eV for energy 
- Forces units will be scaled accordingly 

In [1]:
from pathlib import Path

import numpy as np
import xarray as xr
from ase import Atoms
from ase.io import read, write

# Directory setup. 
ROOT_PATH = Path.cwd()
# Static Grid Files downloaded placed in a folder called static_grid_files
STATIC_GRID_DIR = ROOT_PATH / "static_grid_files"
XYZ_DIR = ROOT_PATH / "xyz_files"
XYZ_DIR.mkdir(exist_ok=True)

In [2]:
# The NetCDF files store quantum chemistry quantities in atomic units.
# ASE/X-MACE convention is Angstrom for positions, eV for energies,
# and eV/Angstrom for forces.
BOHR_TO_ANGSTROM = 0.529177210903
HARTREE_TO_EV = 27.211386245988

def nc_to_extxyz(nc_path, xyz_path):
    """Read one static-grid NetCDF file with xarray and write one X-MACE extended XYZ file."""
    nc_path = Path(nc_path)
    xyz_path = Path(xyz_path)

    with xr.open_dataset(nc_path, engine="netcdf4") as ds:
        # Raw positions are in Bohr; ASE writes positions in Angstrom.
        positions = ds["positions"].values * BOHR_TO_ANGSTROM  # (n_frames, n_atoms, 3)

        # Raw electronic energies are in Hartree; X-MACE losses usually use eV.
        energy = ds["energy"].values * HARTREE_TO_EV  # (n_frames, n_states)

        # Raw forces are Hartree/Bohr; convert to eV/Angstrom.
        forces = ds["forces"].values * HARTREE_TO_EV / BOHR_TO_ANGSTROM  # (n_frames, n_states, n_atoms, 3)

        symbols = [str(symbol) for symbol in ds["symbols"].values.tolist()]

    n_frames = positions.shape[0]
    frames = []
    for frame_idx in range(n_frames):
        atoms = Atoms(symbols=symbols, positions=positions[frame_idx])

        # X-MACE expects energy per geometry as (1, n_states).
        atoms.info["REF_energy"] = energy[frame_idx].reshape(1, -1)

        # NetCDF/xarray stores forces as (states, atoms, xyz).
        # X-MACE expects (atoms, states, xyz).
        atoms.info["REF_forces"] = np.transpose(forces[frame_idx], (1, 0, 2))

        frames.append(atoms)

    write(xyz_path, frames, format="extxyz")
    return xyz_path

In [3]:
converted_xyz_files = []
for nc_path in sorted(STATIC_GRID_DIR.glob("*.nc")):
    xyz_path = XYZ_DIR / f"{nc_path.stem}.xyz"
    converted_xyz_files.append(nc_to_extxyz(nc_path, xyz_path))

print("Converted files:")
for xyz_path in converted_xyz_files:
    frames = read(xyz_path, index=":")
    first = frames[0]
    print(f"- {xyz_path.name}")
    print(f"  frames      : {len(frames)}")
    print(f"  atoms/frame : {len(first)}")
    print(f"  distance 0-1: {np.linalg.norm(first.positions[1] - first.positions[0]):.8f} Angstrom")
    print(f"  REF_energy  : {np.array(first.info['REF_energy']).shape}, first={np.array(first.info['REF_energy'])[0]}")
    print(f"  REF_forces  : {np.array(first.info['REF_forces']).shape}")


Converted files:
- A01_ethene_grid_static.xyz
  frames      : 3731
  atoms/frame : 6
  distance 0-1: 1.12450157 Angstrom
  REF_energy  : (1, 3), first=[-2119.85414173 -2113.86053568 -2109.80020501]
  REF_forces  : (6, 3, 3)
- A02_propene_grid_static.xyz
  frames      : 3731
  atoms/frame : 9
  distance 0-1: 1.05835442 Angstrom
  REF_energy  : (1, 3), first=[-3179.8680406  -3171.9966546  -3165.32449616]
  REF_forces  : (9, 3, 3)
